# Tabular Regression Workflow

This notebook follows an industry-style supervised regression workflow: define the prediction target, build reproducible preprocessing, compare model families, tune hyperparameters, add custom features, and evaluate on an untouched test set.

## Workflow Overview

**Business-style objective:** predict continuous outcomes from tabular data. The main workflow predicts California district median house values; the final section repeats the workflow on hourly bike rental demand.

**Data sources:** California housing data downloaded into `data/housing`; Bike Sharing Demand from OpenML.

**Primary metrics:** RMSE for model selection and test evaluation, with additional MAE/R2 where useful for interpretation.

**Validation strategy:** keep a held-out test set untouched; use cross-validation or validation splits during model selection; refit selected pipelines before final evaluation.

**Production mindset:** transformations are wrapped in Scikit-Learn pipelines so training and inference use the same preprocessing steps.

## Stage 0 - Environment, Data Contract, and Reusable Components

Centralize imports, constants, dataset paths, preprocessing helpers, and custom transformers. This keeps later modeling cells focused on experiments rather than repeated setup code.

These imports and helper functions recreate the applied regression workflow: load the housing data, create the stratified split, build preprocessing pipelines, and define the custom transformers used in the experiments.

In [17]:
from pathlib import Path
import tarfile
import urllib.request

import numpy as np
import pandas as pd
from scipy.stats import randint, loguniform

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.utils.validation import check_array, check_is_fitted

RANDOM_STATE = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
HOUSING_URL = "https://github.com/ageron/data/raw/main/housing.tgz"
HOUSING_CSV = DATA_DIR / "housing" / "housing.csv"
HOUSING_TGZ = DATA_DIR / "housing.tgz"

In [18]:
def load_housing_data():
    """Download if needed, then load the California housing data."""
    DATA_DIR.mkdir(exist_ok=True)
    if not HOUSING_CSV.is_file():
        if not HOUSING_TGZ.is_file():
            urllib.request.urlretrieve(HOUSING_URL, HOUSING_TGZ)
        with tarfile.open(HOUSING_TGZ) as housing_tarball:
            housing_tarball.extractall(path=DATA_DIR)
    return pd.read_csv(HOUSING_CSV)


def split_housing_data(housing):
    """Create the same stratified train/test split used in the chapter."""
    housing = housing.copy()
    housing["income_cat"] = pd.cut(
        housing["median_income"],
        bins=[0.0, 1.5, 3.0, 4.5, 6.0, np.inf],
        labels=[1, 2, 3, 4, 5],
    )
    
    strat_train_set, strat_test_set = train_test_split(
        housing,
        test_size=0.2,
        stratify=housing["income_cat"],
        random_state=RANDOM_STATE,
    )
    for set_ in (strat_train_set, strat_test_set):
        set_.drop("income_cat", axis=1, inplace=True)

    X_train = strat_train_set.drop("median_house_value", axis=1)
    y_train = strat_train_set["median_house_value"].copy()
    X_test = strat_test_set.drop("median_house_value", axis=1)
    y_test = strat_test_set["median_house_value"].copy()
    return X_train, y_train, X_test, y_test


housing_full = load_housing_data()
X_train, y_train, X_test, y_test = split_housing_data(housing_full)

X_train.shape, X_test.shape

((16512, 9), (4128, 9))

In [20]:
def column_ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]


def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler(),
    )


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    """Add RBF similarity features to K-Means cluster centers."""

    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(
            self.n_clusters,
            random_state=self.random_state,
            n_init=10,
        )
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        check_is_fitted(self, "kmeans_")
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return np.array(
            [f"Cluster {i} similarity" for i in range(self.n_clusters)],
            dtype=object,
        )

In [21]:
class KNNIncomeFeature(BaseEstimator, TransformerMixin):
    """Predict median_income from latitude/longitude and output it as one feature.

    The transformer expects three input columns in this order:
    latitude, longitude, median_income.

    It trains KNeighborsRegressor on latitude/longitude to predict median_income.
    During transform(), only latitude/longitude are used as inputs.
    """

    def __init__(self, n_neighbors=5, weights="distance"):
        self.n_neighbors = n_neighbors
        self.weights = weights

    def _to_numpy(self, X):
        if hasattr(X, "to_numpy"):
            return X.to_numpy()
        return np.asarray(X)

    def fit(self, X, y=None):
        X_arr = check_array(self._to_numpy(X), dtype=float, ensure_all_finite=True)
        if X_arr.shape[1] != 3:
            raise ValueError("KNNIncomeFeature expects latitude, longitude, median_income.")
        self.n_features_in_ = X_arr.shape[1]
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)

        self.knn_ = KNeighborsRegressor(
            n_neighbors=self.n_neighbors,
            weights=self.weights,
        )
        self.knn_.fit(X_arr[:, :2], X_arr[:, 2])
        return self

    def transform(self, X):
        check_is_fitted(self, "knn_")
        X_arr = check_array(self._to_numpy(X), dtype=float, ensure_all_finite=True)
        if X_arr.shape[1] != self.n_features_in_:
            raise ValueError(f"Expected {self.n_features_in_} features, got {X_arr.shape[1]}.")
        return self.knn_.predict(X_arr[:, :2]).reshape(-1, 1)

    def get_feature_names_out(self, input_features=None):
        return np.array(["knn_median_income"], dtype=object)

In [23]:
def make_preprocessing(include_knn_income=False):
    """Build the Chapter 2 preprocessing pipeline, optionally with the KNN income feature."""
    log_pipeline = make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(np.log, feature_names_out="one-to-one"),
        StandardScaler(),
    )
    cat_pipeline = make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    )
    cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1.0, random_state=RANDOM_STATE)
    default_num_pipeline = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
    )

    transformers = [
        ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
        ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
        ("people_per_house", ratio_pipeline(), ["population", "households"]),
        (
            "log",
            log_pipeline,
            ["total_bedrooms", "total_rooms", "population", "households", "median_income"],
        ),
        ("geo", cluster_simil, ["latitude", "longitude"]),
        ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
    ]

    if include_knn_income:
        transformers.insert(
            5,
            (
                "knn_income",
                KNNIncomeFeature(n_neighbors=5, weights="distance"),
                ["latitude", "longitude", "median_income"],
            ),
        )

    return ColumnTransformer(
        transformers,
        remainder=default_num_pipeline,
        verbose_feature_names_out=True,
    )


def describe_rmse_scores(scores):
    return pd.Series(scores).describe()

## Stage 1 - SVR Grid Search Baseline

Train a support vector regressor baseline with a small cross-validation grid. The search uses a subset for speed, then the selected pipeline is refit on the full training set before test evaluation.

Train `sklearn.svm.SVR` with linear and RBF kernels. SVMs can be slow on large datasets, so this experiment uses the first 5,000 training instances with 3-fold cross-validation.

In [24]:
X_train_small = X_train.iloc[:5_000]
y_train_small = y_train.iloc[:5_000]

svr_pipeline = Pipeline([
    ("preprocessing", make_preprocessing()),
    ("svr", SVR()),
])

param_grid = [
    {"svr__kernel": ["linear"], "svr__C": [10.0, 30.0, 100.0]},
    {
        "svr__kernel": ["rbf"],
        "svr__C": [30.0, 100.0, 300.0],
        "svr__gamma": [0.01, 0.03, 0.1],
    },
]

grid_search = GridSearchCV(
    svr_pipeline,
    param_grid,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)

grid_search.fit(X_train_small, y_train_small)

svr_grid_rmse = -grid_search.best_score_
grid_search.best_params_, svr_grid_rmse

Fitting 3 folds for each of 12 candidates, totalling 36 fits


({'svr__C': 100.0, 'svr__kernel': 'linear'}, 79441.04503281073)

The grid search above is intentionally run on the first 5,000 training rows so the notebook finishes quickly. That means `grid_search.best_estimator_` was refit on only those 5,000 rows. Before evaluating on the untouched test set, refit a clone of the best pipeline on the full training set.

In [25]:
final_model = clone(grid_search.best_estimator_)
final_model.fit(X_train, y_train)

final_predictions = final_model.predict(X_test)
final_rmse = root_mean_squared_error(y_test, final_predictions)
final_rmse

77065.20245847589

## Stage 2 - Randomized SVR Hyperparameter Search

Use randomized search when the hyperparameter space is wide. This is closer to a practical tuning workflow because it can explore orders of magnitude efficiently.

`RandomizedSearchCV` samples hyperparameter values instead of trying every combination. This is useful when the search space is large.

In [26]:
svr_pipeline = Pipeline([
    ("preprocessing", make_preprocessing()),
    ("svr", SVR()),
])

param_distributions = [
    {
        "svr__kernel": ["linear"],
        "svr__C": loguniform(10.0, 1_000.0),
    },
    {
        "svr__kernel": ["rbf"],
        "svr__C": loguniform(10.0, 1_000.0),
        "svr__gamma": loguniform(0.001, 0.3),
    },
]

rnd_search = RandomizedSearchCV(
    svr_pipeline,
    param_distributions,
    n_iter=12,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

rnd_search.fit(X_train_small, y_train_small)

svr_randomized_rmse = -rnd_search.best_score_
rnd_search.best_params_, svr_randomized_rmse

Fitting 3 folds for each of 12 candidates, totalling 36 fits


({'svr__C': 391.81943471417406, 'svr__kernel': 'linear'}, 74091.76537120686)

## Stage 3 - Model-Based Feature Selection

Add `SelectFromModel` to remove weaker engineered features before fitting the regressor. This tests whether a simpler feature set improves validation performance.

`SelectFromModel` keeps the most important transformed features. Here a random forest estimates feature importances, then another random forest regressor trains on the selected features.

In [27]:
# Use a subset by default to keep the notebook practical. Change to X_train/y_train for a deeper run.
X_train_cv = X_train.iloc[:8_000]
y_train_cv = y_train.iloc[:8_000]

select_pipeline = Pipeline([
    ("preprocessing", make_preprocessing()),
    (
        "feature_selection",
        SelectFromModel(
            RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1),
            threshold="median",
        ),
    ),
    (
        "random_forest",
        RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    ),
])

scores = -cross_val_score(
    select_pipeline,
    X_train_cv,
    y_train_cv,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
)

describe_rmse_scores(scores)

count        3.000000
mean     50339.545002
std       1357.810876
min      48814.653929
25%      49800.460914
50%      50786.267899
75%      51101.990539
max      51417.713179
dtype: float64

In [28]:
select_pipeline.fit(X_train_cv, y_train_cv)
selected_count = select_pipeline.named_steps["feature_selection"].get_support().sum()
total_count = select_pipeline.named_steps["preprocessing"].transform(X_train_cv[:1]).shape[1]
selected_count, total_count

(12, 23)

## Stage 4 - Custom Geographic Feature Engineering

Create a reusable transformer that estimates local median income from nearby districts. The point is to convert domain intuition into a pipeline-safe feature.

This transformer trains a `KNeighborsRegressor` in `fit()` using latitude and longitude to predict `median_income`. In `transform()`, it outputs the KNN-predicted median income as one extra feature.

In [29]:
knn_income_pipeline = Pipeline([
    ("preprocessing", make_preprocessing(include_knn_income=True)),
    (
        "random_forest",
        RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    ),
])

scores = -cross_val_score(
    knn_income_pipeline,
    X_train_cv,
    y_train_cv,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
)

describe_rmse_scores(scores)

count        3.000000
mean     52190.070713
std       1088.685368
min      50938.845599
25%      51824.674945
50%      52710.504291
75%      52815.683270
max      52920.862249
dtype: float64

## Stage 5 - Joint Search Across Preprocessing and Model Choices

Tune preprocessing decisions and model hyperparameters together. In real workflows these choices interact, so they should be validated together when compute allows.

This search tunes both preprocessing and model hyperparameters, including the number of geographic clusters, the RBF gamma, and the KNN income-feature settings.

In [30]:
prep_search_pipeline = Pipeline([
    ("preprocessing", make_preprocessing(include_knn_income=True)),
    (
        "random_forest",
        RandomForestRegressor(n_estimators=120, random_state=RANDOM_STATE, n_jobs=-1),
    ),
])

param_distributions = {
    "preprocessing__geo__n_clusters": randint(5, 40),
    "preprocessing__geo__gamma": loguniform(0.1, 5.0),
    "preprocessing__knn_income__n_neighbors": randint(3, 30),
    "preprocessing__knn_income__weights": ["uniform", "distance"],
    "random_forest__max_features": randint(5, 25),
}

prep_rnd_search = RandomizedSearchCV(
    prep_search_pipeline,
    param_distributions,
    n_iter=10,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

prep_rnd_search.fit(X_train_cv, y_train_cv)

-prep_rnd_search.best_score_, prep_rnd_search.best_params_

Fitting 3 folds for each of 10 candidates, totalling 30 fits


(44578.442286943515,
 {'preprocessing__geo__gamma': 0.432845022129388,
  'preprocessing__geo__n_clusters': 33,
  'preprocessing__knn_income__n_neighbors': 17,
  'preprocessing__knn_income__weights': 'uniform',
  'random_forest__max_features': 12})

## Stage 6 - Reusable Transformer Implementation

Implement a Scikit-Learn-compatible scaler to demonstrate estimator API requirements: `fit`, `transform`, `inverse_transform`, fitted attributes, and feature-name handling.

The clone below supports `fit()`, `transform()`, `inverse_transform()`, `feature_names_in_`, and `get_feature_names_out()`.

In [31]:
class StandardScalerClone(BaseEstimator, TransformerMixin):
    """A small clone of StandardScaler with inverse transform and feature names."""

    def __init__(self, with_mean=True, with_std=True):
        self.with_mean = with_mean
        self.with_std = with_std

    def fit(self, X, y=None):
        if hasattr(X, "columns"):
            self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        X_arr = check_array(X, dtype=float, ensure_all_finite=True)
        self.n_features_in_ = X_arr.shape[1]
        self.mean_ = X_arr.mean(axis=0)
        self.scale_ = X_arr.std(axis=0)
        self.scale_[self.scale_ == 0] = 1.0
        return self

    def transform(self, X):
        check_is_fitted(self)
        X_arr = check_array(X, dtype=float, ensure_all_finite=True)
        self._check_feature_count(X_arr)
        X_scaled = X_arr.copy()
        if self.with_mean:
            X_scaled -= self.mean_
        if self.with_std:
            X_scaled /= self.scale_
        return X_scaled

    def inverse_transform(self, X):
        check_is_fitted(self)
        X_arr = check_array(X, dtype=float, ensure_all_finite=True)
        self._check_feature_count(X_arr)
        X_recovered = X_arr.copy()
        if self.with_std:
            X_recovered *= self.scale_
        if self.with_mean:
            X_recovered += self.mean_
        return X_recovered

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self)
        if input_features is None:
            if hasattr(self, "feature_names_in_"):
                return self.feature_names_in_
            return np.array([f"x{i}" for i in range(self.n_features_in_)], dtype=object)

        input_features = np.asarray(input_features, dtype=object)
        if len(input_features) != self.n_features_in_:
            raise ValueError(
                f"input_features should have length {self.n_features_in_}, got {len(input_features)}."
            )
        if hasattr(self, "feature_names_in_") and not np.array_equal(input_features, self.feature_names_in_):
            raise ValueError("input_features does not match feature_names_in_.")
        return input_features

    def _check_feature_count(self, X):
        if X.shape[1] != self.n_features_in_:
            raise ValueError(f"Expected {self.n_features_in_} features, got {X.shape[1]}.")

In [ ]:
housing_num = housing_full.select_dtypes(include=[np.number]).drop(columns=["median_house_value"])
X_sample = housing_num.iloc[:100]

scaler = StandardScalerClone()
X_scaled = scaler.fit_transform(X_sample)
X_recovered = scaler.inverse_transform(X_scaled)

print("inverse_transform close:", np.allclose(X_sample.to_numpy(), X_recovered))
print("feature_names_in_:", scaler.feature_names_in_)
print("get_feature_names_out():", scaler.get_feature_names_out())

plain_scaler = StandardScalerClone().fit(X_sample.to_numpy())
print("generated names without DataFrame:", plain_scaler.get_feature_names_out())

inverse_transform close: True
feature_names_in_: ['longitude' 'latitude' 'housing_median_age' 'total_rooms'
 'total_bedrooms' 'population' 'households' 'median_income']
get_feature_names_out(): ['longitude' 'latitude' 'housing_median_age' 'total_rooms'
 'total_bedrooms' 'population' 'households' 'median_income']
generated names without DataFrame: ['x0' 'x1' 'x2' 'x3' 'x4' 'x5' 'x6' 'x7']


## Stage 7 - Transfer the Workflow to a Second Regression Dataset

Apply the same regression discipline to a different dataset with time-ordered records. The split respects time order to avoid training on future information.

This workflow repeats the same end-to-end regression process on another dataset. Here we use the Bike Sharing Demand dataset from OpenML. The target is `count`, the number of rented bikes for each hourly record.

Because these rows are time ordered, the split below keeps the first 80% for training and the last 20% for testing. That is more realistic than randomly mixing past and future rows.

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

bike = fetch_openml("Bike_Sharing_Demand", version=2, as_frame=True, parser="auto")
bike_df = bike.frame.copy()
bike_df["count"] = bike_df["count"].astype(float)

bike_df.head()

,season,year,month,hour,holiday,weekday,workingday,weather,temp,feel_temp,humidity,windspeed,count
0,spring,0,1,0,False,6,False,clear,9.84,14.395,0.81,0.0,16.0
1,spring,0,1,1,False,6,False,clear,9.02,13.635,0.80,0.0,40.0
2,spring,0,1,2,False,6,False,clear,9.02,13.635,0.80,0.0,32.0
3,spring,0,1,3,False,6,False,clear,9.84,14.395,0.75,0.0,13.0
4,spring,0,1,4,False,6,False,clear,9.84,14.395,0.75,0.0,1.0


In [ ]:
X_bike = bike_df.drop(columns="count")
y_bike = bike_df["count"].copy()

split_idx = int(len(bike_df) * 0.8)
X_bike_train = X_bike.iloc[:split_idx]
y_bike_train = y_bike.iloc[:split_idx]
X_bike_test = X_bike.iloc[split_idx:]
y_bike_test = y_bike.iloc[split_idx:]

bike_cat_attribs = [
    "season", "year", "month", "hour", "holiday", "weekday", "workingday", "weather"
]

bike_num_attribs = [col for col in X_bike.columns if col not in bike_cat_attribs]

X_bike_train.shape, X_bike_test.shape, bike_cat_attribs, bike_num_attribs

((13903, 12),
 (3476, 12),
 ['season',
  'year',
  'month',
  'hour',
  'holiday',
  'weekday',
  'workingday',
  'weather'],
 ['temp', 'feel_temp', 'humidity', 'windspeed'])

In [35]:
bike_cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore", sparse_output=False),
)
bike_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

bike_preprocessing = ColumnTransformer([
    ("cat", bike_cat_pipeline, bike_cat_attribs),
    ("num", bike_num_pipeline, bike_num_attribs),
])

bike_baseline_model = Pipeline([
    ("preprocessing", bike_preprocessing),
    ("hgb", HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
])

time_cv = TimeSeriesSplit(n_splits=3)
bike_baseline_scores = -cross_val_score(
    bike_baseline_model,
    X_bike_train,
    y_bike_train,
    scoring="neg_root_mean_squared_error",
    cv=time_cv,
    n_jobs=-1,
)

describe_rmse_scores(bike_baseline_scores)

count     3.000000
mean     83.951295
std       8.444765
min      75.779353
25%      79.604586
50%      83.429819
75%      88.037267
max      92.644714
dtype: float64

In [36]:
bike_param_distributions = {
    "hgb__max_iter": randint(100, 350),
    "hgb__max_leaf_nodes": randint(15, 80),
    "hgb__learning_rate": loguniform(0.03, 0.3),
    "hgb__l2_regularization": loguniform(1e-4, 1.0),
}

bike_search = RandomizedSearchCV(
    bike_baseline_model,
    bike_param_distributions,
    n_iter=10,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

bike_search.fit(X_bike_train, y_bike_train)

bike_search_rmse = -bike_search.best_score_
bike_search_rmse, bike_search.best_params_

Fitting 3 folds for each of 10 candidates, totalling 30 fits


(81.84435742079343,
 {'hgb__l2_regularization': 0.00011299516083106625,
  'hgb__learning_rate': 0.26261710554018114,
  'hgb__max_iter': 113,
  'hgb__max_leaf_nodes': 23})

In [37]:
bike_final_model = clone(bike_search.best_estimator_)
bike_final_model.fit(X_bike_train, y_bike_train)

bike_test_predictions = bike_final_model.predict(X_bike_test)
bike_test_rmse = root_mean_squared_error(y_bike_test, bike_test_predictions)
bike_test_mae = mean_absolute_error(y_bike_test, bike_test_predictions)
bike_test_r2 = r2_score(y_bike_test, bike_test_predictions)

pd.Series({
    "test_rmse": bike_test_rmse,
    "test_mae": bike_test_mae,
    "test_r2": bike_test_r2,
})

test_rmse    80.871409
test_mae     56.742053
test_r2       0.865466
dtype: float64

In [38]:
pd.DataFrame({
    "actual_count": y_bike_test.iloc[:10].to_numpy(),
    "predicted_count": bike_test_predictions[:10].round(1),
    "error": (bike_test_predictions[:10] - y_bike_test.iloc[:10].to_numpy()).round(1),
})

,actual_count,predicted_count,error
0,283.0,314.2,31.2
1,253.0,297.9,44.9
2,261.0,292.9,31.9
3,306.0,280.6,-25.4
4,445.0,299.1,-145.9
5,868.0,704.6,-163.4
6,814.0,765.2,-48.8
7,610.0,606.2,-3.8
8,448.0,382.2,-65.8
9,317.0,306.3,-10.7


The final table compares the first few test predictions with the real rental counts. At this point you would inspect errors, try more features or models, tune more carefully, and only then decide whether the model is good enough for the task.

## Workflow Summary and Next Steps

Summarize the modeling decisions, validation results, and where this notebook could be hardened for a production-grade workflow.

The applied regression workflows are implemented above.

For deeper experiments, increase:

- the SVR grid/search ranges,
- `n_iter` in the randomized searches,
- `X_train_cv` from the first 8,000 rows to the full `X_train`,
- the Bike Sharing randomized search ranges or number of iterations.